## tl;dr

PN7C tests whether arrival direction improves next actual-prime ARA-state prediction after controlling shared-gap overlap and one-step raw-gap dynamics. P1–P4 and P7 pass; P5–P6 fail. The local ARA sequence is predictive, but the registered residual-memory core does not pass.


In [1]:
from pathlib import Path
import csv, json, math
HERE = Path.cwd()
results = json.loads((HERE / 'PN7C_ACTUAL_GAP_SEQUENTIAL_MEMORY_RESULTS.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN7C_ACTUAL_GAP_SEQUENTIAL_MEMORY_VALIDATION.json').read_text(encoding='utf-8'))
assert validation['all_passed'] and validation['checks_passed'] == 43
print('Loaded', results['test_id'])
print('Independent validation:', f"{validation['checks_passed']}/{validation['checks_total']}")


Loaded PN7C/ACTUAL-GAP-SEQUENTIAL-MEMORY/CODE-ISOLATED-R11-v1
Independent validation: 43/43


## Context & Methods

For consecutive actual-prime gaps, the native state is `x_i = 2 g_(i+1)/(g_i + g_(i+1))`. ARA-M1 predicts the next binned state from the current state. ARA-M2 adds the previous state, retaining arrival direction. RawGap-M1 uses the exact shared current gap and predicts one raw step before projection through the same ARA bins. Models were frozen on R9–R10 before the PN7C R11 target was constructed.


In [2]:
primary = results['scores']['24']
transfer_gain = primary['ARA-M1']['cross_entropy_bits'] - primary['ARA-M2']['cross_entropy_bits']
observed = results['empirical_target_memory']['24']['conditional_memory_gain_bits']
shuffle_ceiling = max(r['conditional_memory_gain_bits'] for r in results['exact_inventory_shuffles_24_bins'])
markov = results['raw_gap_markov_world_24_bins']['conditional_memory_gain_bits']
assert math.isclose(transfer_gain, 0.11805413043323387, rel_tol=0, abs_tol=1e-12)
print(f'ARA transferred arrival gain: {transfer_gain:.6f} bits/read')
print(f'Observed minus shuffle ceiling: {observed-shuffle_ceiling:.6f} bits')
print(f'Observed minus raw Markov world: {observed-markov:.6f} bits')


ARA transferred arrival gain: 0.118054 bits/read
Observed minus shuffle ceiling: 0.020467 bits
Observed minus raw Markov world: 0.003051 bits


## Data

Development uses exact actual-prime gaps from R9 and R10. Evaluation uses 39,475,590 internal R11 gaps, producing 39,475,587 scored three-state events. Window boundaries are not joined. R11 is code-isolated for PN7C but historically opened.


In [3]:
development = json.loads((HERE / 'PN7C_DEVELOPMENT_GAPS.json').read_text(encoding='utf-8'))
target = json.loads((HERE / 'PN7C_R11_TARGET_GAPS.json').read_text(encoding='utf-8'))
assert all(v['matches'] for v in development['reconciliation'].values())
assert target['prime_count_reconciliation']['matches']
assert target['gap_count'] == 39_475_590 and results['target']['scored_events'] == 39_475_587
print('Development prime counts:', {k: v['prime_count'] for k, v in development['rungs'].items()})
print('R11 prime/gap counts:', target['prime_count'], target['gap_count'])


Development prime counts: {'r9': 482449, 'r10': 4341930}
R11 prime/gap counts: 39475591 39475590


## Results

The table below separates prediction from structural controls. Lower cross-entropy is better.


In [4]:
print('Model          CE bits    Brier      Top-3')
for name in ('ARA-IID', 'ARA-M1', 'ARA-M2', 'RawGap-M1'):
    row = primary[name]
    print(f"{name:12s} {row['cross_entropy_bits']:8.6f}  {row['brier_score']:8.6f}  {row['top3_accuracy']:8.4%}")
print('\nRegistered criteria:')
for key, row in results['criteria'].items():
    print(key, 'PASS' if row['passed'] else 'FAIL')
assert [results['criteria'][f'P{i}']['passed'] for i in range(1, 8)] == [True, True, True, True, False, False, True]
assert not results['residual_ordered_memory_core_passed']


Model          CE bits    Brier      Top-3
ARA-IID      4.496676  0.953978  17.9110%
ARA-M1       4.230639  0.939698  25.3111%
ARA-M2       4.112585  0.933152  27.9523%
RawGap-M1    3.621871  0.907127  36.3158%

Registered criteria:
P1 PASS
P2 PASS
P3 PASS
P4 PASS
P5 FAIL
P6 FAIL
P7 PASS


## Takeaways

Arrival direction is a stable, transferable part of the local ARA relation, and real gap order exceeds overlap-only shuffles. The effect is almost reproduced by a first-order exact-gap Markov world, and the exact raw-gap predictor beats the compressed ARA model. This supports local ARA sequential structure but not a distinct residual memory law. PN7C does not test a slow adult wave extending across many primes.
